# Olist Brazilian E-Commerce Analytics
## Project 1 · P1-02 — Data Cleaning, Validation & Controlled Merge

**Purpose:** clean and combine the orders and customers tables without silently changing the analytical population or damaging the meaning of the data.

**Working environment:** Jupyter Notebook in VS Code · Python · Pandas · DuckDB

**Selected source tables:**
- `olist_orders_dataset.csv`
- `olist_customers_dataset.csv`

**Task boundary:** this notebook covers source loading, structural inspection, timestamp parsing, duplicate/key checks, a controlled merge, row-count reconciliation, and CSV export. It does not perform full EDA, business KPI analysis, or broad feature engineering.

> **Strict instructor rule:** write the logic yourself in every `# TODO` cell. Do not remove rows merely to make a validation check pass. A failed check is evidence to investigate, not an invitation to hide the problem.


## 0. Operating principles and data contract

### Non-negotiable conventions

- Treat files in `data/raw/` as immutable source evidence. Never overwrite them.
- Keep raw and cleaned DataFrames separate. Use names that reveal the table and processing state.
- Use `snake_case` for variable names and preserve the source column names unless a documented transformation requires otherwise.
- Record row counts before and after every operation that can change rows.
- A missing value is not automatically a bad record. Interpret missingness in the context of the business process and `order_status`.
- Never use `drop_duplicates()` as a reflex. First establish whether the duplicate is an exact repeated row, a conflicting record for the same key, or a legitimate repeated business event.
- Do not use `inner` join just to get a tidy result. Choose the join type based on the population you must preserve.

### Selected table grain and relationship

| Table | One row represents | Key | Important distinction |
|---|---|---|---|
| `orders` | One order | `order_id` | Contains order status and lifecycle timestamps |
| `customers` | One order-specific customer record | `customer_id` | `customer_id` is assigned per order; `customer_unique_id` identifies the real buyer across orders |

**Expected relationship:** `orders.customer_id` ↔ `customers.customer_id` is one-to-one in the original Olist model. Verify this against the actual files before merging; never assume the downloaded copy is intact.

**Critical identity rule:** `customer_unique_id` is allowed to repeat. Repetition there represents the same real buyer placing multiple orders. Do not deduplicate the customers table using `customer_unique_id` for this task.

### Why this pair is appropriate

The orders table gives you five lifecycle timestamps to parse. The customers table enriches each order with geographic attributes. Because both tables are at order-specific grain, a one-to-one merge should preserve the orders population exactly.


## 1. Environment and imports

Use the existing project environment. Do not upgrade packages inside this notebook just to match an online example. Record the versions actually used when you run the notebook.

- **Pathlib:** constructs portable paths and helps keep raw and processed data locations explicit.
- **Pandas:** loads, inspects, cleans, validates, merges, and exports the two tables.
- **DuckDB:** available for optional independent SQL checks later in the project; this exercise is intentionally centred on Pandas, so do not duplicate every Pandas operation in DuckDB.


In [ ]:
# Standard project imports — setup only; implement the analytical logic in the TODO cells.
from pathlib import Path  # Portable file and directory path handling
import pandas as pd  # Tabular data loading, cleaning, validation, merging, and export
import duckdb  # Optional SQL engine for independent relational checks elsewhere in the project


## 2. Define source and output paths

Keep raw inputs and generated outputs in separate locations. Use project-relative paths rather than hard-coded machine-specific absolute paths.

**Expected project convention:**

- Raw inputs: `data/raw/`
- Processed outputs: `data/processed/`

Do not move or rename the source files just to make the notebook work. Align these paths with the structure of your existing repository.


In [ ]:
# TODO: Define Path objects for the raw data directory, orders CSV, customers CSV,
#       processed output directory, and final cleaned CSV path.
# TODO: Check that both input files exist before attempting to load them.
# TODO: Create the processed output directory if it does not already exist.


## 3. Load the two source tables independently

Load each CSV into its own raw DataFrame. Do not merge during ingestion.

**Professional considerations:**
- Preserve identifiers as strings. IDs are labels, not quantities; converting them to integers can destroy leading zeros in identifiers such as postal-code prefixes.
- Do not parse timestamps during `read_csv` for this exercise. Load the raw values first so that parsing failures can be measured and explained explicitly.
- Avoid arbitrary row filters or `nrows` limits. The validation must cover the full source population.
- Keep the raw DataFrames unchanged and create working copies for transformations.


In [ ]:
# TODO: Load the orders CSV into a raw DataFrame named orders_raw.
# TODO: Load the customers CSV into a raw DataFrame named customers_raw.
# TODO: Confirm both objects are Pandas DataFrames and retain all source columns.


## 4. Establish the baseline: shape, schema, and row counts

Run `info()` on both tables and capture their shapes. The baseline is the reference point against which every later row-count change must be reconciled.

For each table, document:
- Initial row count and column count.
- Column names and data types.
- Memory usage.
- Whether the expected key columns are present.

`DataFrame.info()` prints a summary; it does not return a normal tabular result. Keep its output visible in the notebook and separately record the row counts you will use in your audit log.


In [ ]:
# TODO: Run info() on orders_raw and customers_raw.
# TODO: Capture the initial row count and column count for each DataFrame.
# TODO: Create a small baseline summary showing table_name, row_count, and column_count.
# TODO: Record the Pandas version used for this run.


## 5. Confirm table grain and key integrity before cleaning

Do not infer grain from the number of rows. Validate the business key.

### Required key checks

**Orders**
- Candidate primary key: `order_id`.
- Check whether it is present, non-null, and unique.

**Customers**
- Candidate primary key for this source table: `customer_id`.
- Check whether it is present, non-null, and unique.
- Do not expect `customer_unique_id` to be unique; it identifies a real buyer across potentially multiple order-specific customer IDs.

A primary-key duplicate is not the same thing as an exact duplicate row. A repeated key with different attribute values is a conflict that needs investigation, not an automatic deletion.


In [ ]:
# TODO: Check the required key columns exist in their respective DataFrames.
# TODO: For orders, measure null order_id values and duplicate order_id values.
# TODO: For customers, measure null customer_id values and duplicate customer_id values.
# TODO: Count distinct customer_unique_id values and compare with customers row count.
# TODO: Explain why repeated customer_unique_id values can be legitimate.


## 6. Profile missing values before making cleaning decisions

Use `isna().sum()` to quantify missing values by column in both tables. Also calculate missing-value percentages so that large and small tables can be compared meaningfully.

### Interpret missingness, do not merely count it

Several order lifecycle timestamps can legitimately be missing when an order has not reached that stage or was cancelled. For example, a missing customer-delivery timestamp on a cancelled order is not equivalent to a malformed timestamp on a delivered order.

Do not drop orders because one or more lifecycle timestamps are null. First compare missingness with `order_status` and the meaning of the field. The estimate timestamp may have a different completeness expectation from actual delivery timestamps.

For this task, the goal is to document missingness and distinguish expected business-process nulls from suspicious data-quality issues. Do not introduce imputation unless you have a defensible business rule.


In [ ]:
# TODO: Calculate null counts and null percentages for every column in both tables.
# TODO: Inspect missing values in the five order timestamp columns alongside order_status.
# TODO: Identify which missing values are expected from the order lifecycle and which need investigation.
# TODO: Record the rationale; do not drop rows just because a field contains nulls.


## 7. Parse order lifecycle timestamps with an auditable failure count

The following columns in `orders` are timestamp fields:

- `order_purchase_timestamp`
- `order_approved_at`
- `order_delivered_carrier_date`
- `order_delivered_customer_date`
- `order_estimated_delivery_date`

### Parsing policy

1. Inspect representative raw values and confirm their consistent format.
2. Prefer an explicit `format=` when the source values follow a known, consistent format. Avoid relying on ambiguous date inference.
3. Use `pd.to_datetime(..., errors="coerce")` so unparseable values become `NaT` and can be counted rather than crashing the workflow.
4. Preserve enough raw evidence to distinguish pre-existing nulls from newly failed parses.
5. Do not set `utc=True` or localize timezone-naive values without evidence of the source timezone. A timezone assumption changes the interpretation of a timestamp.

### Critical distinction: pre-existing null versus parse failure

`errors="coerce"` produces `NaT` for invalid values, but the source may already contain nulls. For each column, calculate separately:

- **Pre-existing missing:** raw value was null before parsing.
- **New parse failure:** raw value was non-null but the parsed value is `NaT`.
- **Total missing after parsing:** parsed value is `NaT` for any reason.

A parse failure count of zero is not proof that every timestamp exists; legitimate source nulls may remain.


In [ ]:
# TODO: Inspect representative non-null raw values from each timestamp column.
# TODO: Decide and document the exact parsing format based on the observed source representation.
# TODO: Keep raw source columns available for audit, then parse the five timestamp columns
#       in a working orders DataFrame with pd.to_datetime(..., errors="coerce").
# TODO: For each timestamp, report pre-existing nulls, newly created NaT values,
#       total NaT values after parsing, and the resulting dtype.
# TODO: Investigate any newly created NaT values; do not silently accept them.


## 8. Duplicate audit: exact rows versus duplicate business keys

Use two distinct checks on each table:

1. **Exact duplicate rows:** all column values match an earlier row (`duplicated()` with no subset).
2. **Duplicate key values:** the candidate primary key appears more than once (`duplicated(subset=[key])`).

These checks answer different questions. An exact duplicate may be a repeated copy of the same record. A duplicate primary key can represent conflicting records and is more serious.

### Removal policy

- First record the row count and duplicate count.
- Inspect the affected rows and determine whether they are truly redundant.
- Remove only verified exact duplicate copies when the business meaning supports doing so.
- If a key occurs more than once with differing attributes, do not keep the first or last row arbitrarily. Investigate, document, and retain the evidence until a deterministic resolution rule exists.
- Never remove repeated `customer_unique_id` values as duplicates for this task.

Every removal must have a documented reason and a before/after row count.


In [ ]:
# TODO: Count exact duplicate rows in the working orders and customers DataFrames.
# TODO: Count duplicate candidate primary keys: orders.order_id and customers.customer_id.
# TODO: Inspect all rows involved in any duplicate-key group and classify them as exact repeats
#       or conflicting records.
# TODO: Apply only justified duplicate removal, if needed; otherwise retain all rows.
# TODO: Record row counts before and after each actual removal and document the rationale.


## 9. Build a cleaning decision and row-count audit log

A production-quality cleaning workflow should make every population change explainable.

Your audit log should have one row per meaningful processing stage, with fields such as:

| Field | Meaning |
|---|---|
| `stage` | Name of the operation |
| `table_name` | Table or output affected |
| `rows_before` | Row count immediately before the operation |
| `rows_after` | Row count immediately after the operation |
| `rows_removed` | Difference, where applicable |
| `reason` | Business or data-quality justification |
| `decision` | Retained, removed, quarantined, or no action |

Timestamp parsing should not change the number of rows. Null profiling should not change the number of rows. A duplicate-removal step should change the count only when a justified removal actually occurs.


In [ ]:
# TODO: Create a structured cleaning audit log for both tables.
# TODO: Include baseline counts, timestamp parsing (expected row-neutral), duplicate handling,
#       and any other transformation that changes rows.
# TODO: For every non-zero row-count change, state exactly which records were affected and why.


## 10. Validate join keys and referential coverage before merging

Before the merge, test the relationship explicitly.

### Expected relationship

`orders.customer_id` → `customers.customer_id`

For the original Olist source, `customer_id` is generated per order. Therefore, the expected relationship between these two order-grain tables is **one-to-one**. The actual-person identifier is `customer_unique_id`, which may repeat.

### Pre-merge checks

- `orders.order_id` is unique and non-null.
- `customers.customer_id` is unique and non-null.
- Check null values in both `customer_id` join-key columns.
- Measure orders whose `customer_id` has no matching customer record.
- Measure customer records that have no matching order record, and decide whether they should be retained in a separate exception report.

**Pandas-specific caution:** unlike conventional SQL joins, Pandas can match null merge keys to each other. Do not let null keys silently match. Profile them and handle them explicitly before calling `merge()`.

If the keys are not unique as expected, stop and investigate. Do not switch the validation to `many_to_many` just to make the merge run.


In [ ]:
# TODO: Confirm key uniqueness and non-nullness immediately before merging.
# TODO: Measure unmatched orders and unmatched customer records using customer_id.
# TODO: Inspect any null join keys and define an explicit treatment before merging.
# TODO: State whether the observed data satisfies the expected one-to-one relationship.


## 11. Select and justify the merge strategy

### Recommended strategy for this task

- **Left table:** cleaned orders.
- **Right table:** cleaned customers.
- **Join key:** `customer_id`.
- **Join type:** `left`.
- **Cardinality validation:** `one_to_one`.
- **Merge indicator:** enabled during validation so you can count `both`, `left_only`, and `right_only` outcomes.

### Why a left join?

The analytical population for this exercise is all orders. A left join preserves every row from the orders table, even if a customer record is missing. An inner join would silently remove unmatched orders; an outer join would add customer-side records without a corresponding order to an order-grain output.

### Why one-to-one validation?

The `one_to_one` validation checks that the join key is unique on both inputs. This is the expected Olist relationship because the `customer_id` in the customers table is order-specific. The validation is a guardrail, not a substitute for understanding the model.

### Join types in context

| Join type | What it preserves | Why it is / is not suitable here |
|---|---|---|
| `inner` | Only matching keys | Not the default: can hide unmatched orders by dropping them |
| `left` | All rows from orders plus matches from customers | Recommended: preserves the order population |
| `outer` | All keys from both sides | Useful for full reconciliation, but not the desired order-grain deliverable |

If your checks show that the actual files do not meet the expected one-to-one relationship, do not force this merge. Diagnose the source discrepancy first.


In [ ]:
# TODO: Merge cleaned orders with cleaned customers using customer_id.
# TODO: Use how="left", validate="one_to_one", and indicator=True (or a named indicator column).
# TODO: Keep the merge indicator until unmatched-record checks are complete.
# TODO: Do not write the merge until the pre-merge key checks have been reviewed.


## 12. Post-merge reconciliation and integrity checks

A successful `merge()` call does not prove the result is correct. Reconcile the output.

### Required post-merge checks

1. Compare the orders row count immediately before merge with the merged row count.
2. Confirm `order_id` remains unique in the merged result.
3. Confirm the merge indicator distribution and explain any `left_only` records.
4. Confirm no unexpected customer-only records entered the left-joined order output.
5. Check overlapping column names and suffixes; ensure the output does not contain ambiguous duplicate attributes.
6. Recheck key nulls and critical column dtypes.

**Expected outcome:** with valid unique keys on both inputs and no row filtering, a left one-to-one merge should preserve the orders row count. An unchanged row count is necessary but not sufficient: you must also inspect match coverage and key uniqueness.

Do not drop unmatched orders merely to achieve 100% matching. Record them as exceptions and explain the impact.


In [ ]:
# TODO: Compare pre-merge orders rows with post-merge rows and explain any difference.
# TODO: Verify order_id uniqueness in the merged DataFrame.
# TODO: Summarize merge-indicator categories and calculate unmatched-order coverage.
# TODO: Inspect and resolve ambiguous overlapping column names without discarding source evidence.
# TODO: Confirm the final DataFrame remains at one row per order.


## 13. Export the cleaned merged dataset

Export to a new file under `data/processed/`. Never overwrite either raw input.

### Export requirements

- Use a descriptive filename that records the project and task.
- Use `index=False`; the Pandas row index is not a business column.
- Keep the merge indicator out of the analytical output after reconciliation, unless you deliberately retain it as a documented audit field.
- Be aware that CSV does not preserve Pandas datetime dtypes. The timestamp values are serialized as text; downstream consumers must parse them again or use a typed format such as Parquet in a later stage.

The required deliverable for this task is CSV. Do not silently substitute another file format.


In [ ]:
# TODO: Remove the temporary merge-indicator field from the final analytical DataFrame
#       only after its reconciliation results have been documented.
# TODO: Export the final cleaned DataFrame to the processed CSV path with index=False.
# TODO: Confirm the output file exists and report its file size.
# TODO: Read the exported CSV back for a lightweight verification of row count and column names.
# TODO: Explain why datetime columns require parsing again after a CSV round trip.


## 14. Final cleaning and merge report

Write a concise, evidence-based report using the counts produced by your notebook. Do not hard-code expected counts from an online tutorial; use the counts from the exact source files you loaded.

Your report must state:

- Source file names and baseline row counts.
- Grain and candidate key for each table.
- Missing-value findings, distinguishing expected lifecycle nulls from suspicious values.
- Timestamp parse failures, separately from pre-existing nulls.
- Exact duplicate counts and duplicate-key findings.
- Every row removal, if any, with a defensible reason.
- Join key, join type, cardinality validation, and unmatched-key counts.
- Pre-merge and post-merge row counts.
- Final output path and exported row count.

A clean result may legitimately involve **zero rows removed**. Professional cleaning is not measured by how many rows you delete; it is measured by whether every transformation is justified and validated.


In [ ]:
# TODO: Produce a compact final report from the audit log and validation results.
# TODO: Explicitly state whether any records were removed. If none, explain why retaining them
#       was the correct data-quality decision.
# TODO: State whether the final output is still at one row per order.


## 15. Guide — explain the two tables and their grain

Answer in your own words, using the evidence from your validation cells:

1. Which two Olist tables did you use?
2. What does one row in `olist_orders_dataset` represent?
3. What does one row in `olist_customers_dataset` represent?
4. Why is `customer_id` the correct join key for these two tables?
5. Why is `customer_unique_id` not the correct key for this order-level merge, and why can it repeat?
6. What cardinality did you validate, and what does that validation protect against?
7. Why should a left join preserve the orders row count in this design?

**Professional terminology to use accurately:** grain, primary key, foreign key, one-to-one relationship, referential integrity, unmatched records, and row-count reconciliation.


In [ ]:
# TODO: Write a concise, technically accurate answer to the Guide questions above.
# TODO: Reference the observed row counts and key-validation results from your own run.


## 16. G01 hiring-gate quality checklist

Before considering this task complete, verify each item with evidence in the notebook.

- [ ] I can state the grain of both tables without confusing an order-specific customer record with a real person.
- [ ] I recorded baseline row and column counts before cleaning.
- [ ] I inspected schemas and confirmed the relevant key and timestamp columns.
- [ ] I measured nulls and interpreted lifecycle nulls in business context.
- [ ] I separated exact duplicate rows from duplicate primary-key values.
- [ ] I never deleted conflicting key records without an explicit resolution rule.
- [ ] I reported pre-existing timestamp nulls separately from newly created `NaT` values.
- [ ] I validated uniqueness and nullness of both join keys before merging.
- [ ] I used a left join and explicitly validated the expected one-to-one cardinality.
- [ ] I reconciled pre-merge and post-merge row counts and checked unmatched records.
- [ ] I can justify every row removed, or explain why no rows were removed.
- [ ] I exported a new CSV without overwriting raw inputs and verified the exported row count.
- [ ] My notebook can be re-run from top to bottom in the intended project environment.

**Completion standard:** every check is supported by an observed result or a documented exception. Do not mark a checkbox complete merely because the code executed without an error.
